In [23]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [18]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [26]:
def calculate_claude_haiku_price(input_tokens, output_tokens):
    # Prices per 1M tokens (Claude 3 Haiku)
    INPUT_PRICE_PER_MILLION = 0.25   # $0.25 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 1.25  # $1.25 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_claude_haiku_price(652, 33)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.00020425


In [24]:
from rag_helper import RAGBase
from anthropic import Anthropic

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

anthropic_client = Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY")
)
assistant = RAGBase(
    index=index,
    llm_client=anthropic_client,
    instructions=instructions,
)

In [8]:
from anthropic import Anthropic
import os

client = Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY")
)


In [14]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [9]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [20]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        message = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1000,
        messages=messages
        )

        messages.extend(message.content)

        for item in message.content:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [25]:
agent_loop(instructions, "How do I run Ollama locally bro I'm struggeling sooo hard")

iteration #1...


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages: Unexpected role "developer". Allowed roles are "user" or "assistant"'}, 'request_id': 'req_011CcmESHXG6jDA3RUtHFgZt'}